<a href="https://colab.research.google.com/github/rithishdev/DL_Lab/blob/main/DL_Exp10.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

24BAD099 - Rithish A

In [1]:
%%bash
pip install -q transformers torch gradio

In [1]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, pipeline

# Roll No: Student_Roll_Number_Here
print("Initializing conversational model and tokenizer...")

# Selecting an efficient instruction-tuned model suitable for Google Colab runtimes
model_name = "HuggingFaceTB/SmolLM2-135M-Instruct"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype=torch.bfloat16 if torch.cuda.is_available() else torch.float32,
    device_map="auto"
)

# Configure pipeline for text generation
chatbot_pipeline = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer
)

print("Model loaded successfully!")

Initializing conversational model and tokenizer...


config.json:   0%|          | 0.00/861 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/3.76k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/801k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/466k [00:00<?, ?B/s]

...

model.safetensors: reconstructing file:   0%|          |  0.00B /  269MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/272 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/132 [00:00<?, ?B/s]

Model loaded successfully!


In [2]:
def generate_response(user_message, history=None):
    """
    Accepts a user message, processes it using Chat Templates for the HuggingFace model,
    and returns the model response. Maintains basic validation and exception handling.
    """
    if history is None:
        history = []

    # Handle empty input
    if not user_message or str(user_message).strip() == "":
        return "Please enter a valid message!", history

    try:
        # Format history and prompt using conversational template
        messages = []
        for user_chat, bot_chat in history:
            messages.append({"role": "user", "content": user_chat})
            messages.append({"role": "assistant", "content": bot_chat})

        messages.append({"role": "user", "content": user_message.strip()})

        # Prepare prompt
        prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)

        # Generate response
        outputs = chatbot_pipeline(
            prompt,
            max_new_tokens=150,
            do_sample=True,
            temperature=0.7,
            top_k=50,
            top_p=0.95,
            pad_token_id=tokenizer.eos_token_id
        )

        full_text = outputs[0]["generated_text"]
        # Extract the assistant's response from template
        response = full_text[len(prompt):].strip()

        # Update history
        history.append((user_message, response))
        return response, history

    except Exception as e:
        return f"An error occurred during response generation: {str(e)}", history

# Basic pipeline test with one prompt (Part A Objective)
test_response, test_history = generate_response("Hello, who are you?")
print(f"Test Prompt: Hello, who are you?\nResponse: {test_response}")

[transformers] Passing `generation_config` together with generation-related arguments=({'temperature', 'top_k', 'pad_token_id', 'do_sample', 'top_p', 'max_new_tokens'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
[transformers] Both `max_new_tokens` (=150) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer GPT2Tokenizer. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup any

Test Prompt: Hello, who are you?
Response: Hello, I am a user named SmolLM, a helpful AI assistant who is trained on the vast knowledge base of the Human Resources (HR) department at Hugging Face. I can provide assistance with a wide range of tasks, including answering questions, explaining processes, and helping with career development.


In [ ]:
import gradio as gr
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, pipeline

# Fallback initialization to guarantee all required objects exist inside this cell
if 'chatbot_pipeline' not in globals() or 'tokenizer' not in globals():
    print("Initializing fallback model and tokenizer inside Gradio cell...")
    model_name = "HuggingFaceTB/SmolLM2-135M-Instruct"
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    model = AutoModelForCausalLM.from_pretrained(
        model_name,
        torch_dtype=torch.bfloat16 if torch.cuda.is_available() else torch.float32,
        device_map="auto"
    )
    chatbot_pipeline = pipeline("text-generation", model=model, tokenizer=tokenizer)

def self_contained_generate_response(user_message, history=None):
    if history is None:
        history = []
    if not user_message or str(user_message).strip() == "":
        return "Please enter a valid message!", history
    try:
        messages = []
        for user_chat, bot_chat in history:
            messages.append({"role": "user", "content": user_chat})
            messages.append({"role": "assistant", "content": bot_chat})
        messages.append({"role": "user", "content": user_message.strip()})

        prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
        outputs = chatbot_pipeline(
            prompt,
            max_new_tokens=150,
            do_sample=True,
            temperature=0.7,
            top_k=50,
            top_p=0.95,
            pad_token_id=tokenizer.eos_token_id
        )
        full_text = outputs[0]["generated_text"]
        response = full_text[len(prompt):].strip()
        history.append((user_message, response))
        return response, history
    except Exception as e:
        return f"An error occurred during response generation: {str(e)}", history

def gradio_chat_interface(user_message, history):
    """
    Adapter function for Gradio chatbot tracking.
    """
    formatted_history = []
    for chat in history:
        if len(chat) >= 2 and chat[0] is not None and chat[1] is not None:
            formatted_history.append((chat[0], chat[1]))

    # Fixed: Call self_contained_generate_response explicitly
    response, updated_history = self_contained_generate_response(user_message, formatted_history)
    return response

# Close any existing active Gradio instances to release the ports correctly
try:
    gr.close_all()
except Exception:
    pass

# Building the interface using Gradio's simple ChatInterface
demo = gr.ChatInterface(
    fn=gradio_chat_interface,
    title="AI Student Chatbot (SmolLM2)",
    description="Interact with a lightweight Transformer-based Large Language Model. Developed for Lab Assignment (Roll No:24BAD099, Name:Rithish A).",
    examples=[
        "Hello, tell me a quick joke!",
        "What is the distance between the Earth and the Moon?",
        "Write a Python program to reverse a list."
    ]
)

# Launching with public share link for remote deployment/access
demo.launch(share=True, debug=True)

Closing server running on port: 7860
Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://fc0dbfe8edc0b209c4.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [4]:
# Part B: Testing with five different prompts
prompts = [
    "What is the capital of France?",  # Factual
    "Can you explain what a Large Language Model is in simple terms?",  # Informational
    "Write a short email template requesting a day off work.",  # Task-oriented
    "What is your favorite book, and why do you like it?",  # Conversational / Creative
    "If a tree falls in a forest and no one is around to hear it, does it make a sound?"  # Philosophical
]

print("=== Starting Prompts Evaluation ===\n")
for i, prompt in enumerate(prompts, 1):
    print(f"Prompt {i}: {prompt}")
    response, _ = generate_response(prompt, history=[])
    print(f"Response {i}: {response}\n" + "-"*50 + "\n")

[transformers] Both `max_new_tokens` (=150) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


=== Starting Prompts Evaluation ===

Prompt 1: What is the capital of France?


[transformers] Both `max_new_tokens` (=150) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Response 1: The capital of France is Paris. Paris is a city that has a rich history and is known for its historical landmarks, culture, and iconic attractions. It is the largest city in France, covering an area of approximately 800 square kilometers, making it the most populous city in France. Paris is also a global center for fashion, gastronomy, and culture, attracting millions of tourists each year.
--------------------------------------------------

Prompt 2: Can you explain what a Large Language Model is in simple terms?


[transformers] Both `max_new_tokens` (=150) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Response 2: In simple terms, a Large Language Model is a type of AI model that uses a vast amount of data and vast amounts of training data to make predictions or decisions. It's like a supercomputer that processes a huge amount of information to come up with an answer. This model is based on a large dataset of text data, often called text-based data, to train it. The more text data is fed into the model, the more accurate and reliable the predictions become.

Think of it like a massive library where every book or text has been cataloged and organized. The model, in this case, is like a librarian who collects all the books and organizes them to make it easier to find what you're looking for. The larger the library
--------------------------------------------------

Prompt 3: Write a short email template requesting a day off work.


[transformers] Both `max_new_tokens` (=150) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Response 3: Subject: Please Don't Forced to Work on Day Off

Dear [Name],

I hope this email finds you well. I need your help with a day off. I'm currently working on a project for [Project Name], and I've been putting in a lot of extra hours. I've been feeling quite exhausted and would love some time to unwind and recharge.

Could you please suggest a day off I can take?

Sincerely,
[Your Name]
--------------------------------------------------

Prompt 4: What is your favorite book, and why do you like it?


[transformers] Both `max_new_tokens` (=150) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Response 4: I'm delighted to share my favorite book with you. There are so many incredible stories, characters, and themes that captivate my heart. But if I'm being honest, my favorite book is probably "The Brief History of Time" by Stephen Hawking. His insight into the universe, his simplicity, and his profound understanding of the human experience make it an absolute joy to read.

The book is a thought-provoking exploration of the concept of time and its implications, and it's a testament to the power of human curiosity and creativity. Hawking's writing style is lyrical, yet accessible, and his explanations of complex scientific concepts are clear and concise. It's a book that leaves you with a lasting impression, and I highly recommend it to anyone interested
--------------------------------------------------

Prompt 5: If a tree falls in a forest and no one is around to hear it, does it make a sound?
Response 5: No, the sound of the tree falling in the forest does not make a sound.